# Multi-Agent Environments

Two multiagent environments are given in the package:

* [GeneralSatelliteTasking](../api_reference/index.rst#bsk_rl.GeneralSatelliteTasking), 
  a [Gymnasium](https://gymnasium.farama.org)-based environment and the basis for all other environments.
* [ConstellationTasking](../api_reference/index.rst#bsk_rl.ConstellationTasking), which
  implements the [PettingZoo parallel API](https://pettingzoo.farama.org/api/parallel/).

The latter is preferable for multi-agent RL (MARL) settings, as most algorithms are designed
for this kind of API.

## Configuring the Environment

For this example, a multisatellite target imaging environment will be used. The goal is
to maximize the value of unique images taken.

As usual, the satellite type is defined first.

In [1]:
from typing import ClassVar

from bsk_rl import act, comm, data, obs, sats, scene
from bsk_rl.sim import dyn, fsw


class ImagingSatellite(sats.ImagingSatellite):
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.OpportunityProperties(
            dict(prop="priority"),
            dict(prop="opportunity_open", norm=5700.0),
            n_ahead_observe=10,
        )
    ]
    action_spec: ClassVar[list[act.Action]] = [act.Image(n_ahead_image=10)]
    dyn_type = dyn.FullFeaturedDynModel
    fsw_type = fsw.SteeringImagerFSWModel

Satellite properties are set to give the satellite near-unlimited power and storage resources. To randomize some parameters in a correlated manner across satellites, a ``sat_arg_randomizer`` is set and passed to the environment. In this case, the satellites are distributed in a trivial single-plane Walker-delta constellation.

In [2]:
from bsk_rl.utils.orbital import walker_delta_args

sat_args = dict(
    imageAttErrorRequirement=0.01,
    imageRateErrorRequirement=0.01,
    batteryStorageCapacity=1e9,
    storedCharge_Init=1e9,
    dataStorageCapacity=1e12,
    u_max=0.4,
    K1=0.25,
    K3=3.0,
    omega_max=0.087,
    servo_Ki=5.0,
    servo_P=150 / 5,
)
sat_arg_randomizer = walker_delta_args(altitude=800.0, inc=60.0, n_planes=1)

## Gym API

GeneralSatelliteTasking uses tuples of actions and observations to interact with the
environment.

In [3]:
from bsk_rl import GeneralSatelliteTasking

env = GeneralSatelliteTasking(
    satellites=[
        ImagingSatellite("EO_1", sat_args),
        ImagingSatellite("EO_2", sat_args),
        ImagingSatellite("EO_3", sat_args),
    ],
    scenario=scene.UniformTargets(1000),
    rewarder=data.UniqueImageReward(),
    communicator=comm.LOSCommunication(),  # Note that dyn must inherit from LOSCommunication
    sat_arg_randomizer=sat_arg_randomizer,
    log_level="INFO",
)
env.reset()

env.observation_space

2026-11-04 04:33:42,877 gym                            INFO       Resetting environment with seed=4017602697


2026-11-04 04:33:42,878 scene.targets                  INFO       Generating 1000 targets


2026-11-04 04:33:42,932 sats.satellite.EO_1            INFO       <0.00> EO_1: Finding opportunity windows from 0.00 to 600.00 seconds


2026-11-04 04:33:42,951 sats.satellite.EO_2            INFO       <0.00> EO_2: Finding opportunity windows from 0.00 to 600.00 seconds


2026-11-04 04:33:42,967 sats.satellite.EO_3            INFO       <0.00> EO_3: Finding opportunity windows from 0.00 to 600.00 seconds


2026-11-04 04:33:42,982 gym                            INFO       <0.00> Environment reset


Tuple(Box(-1e+16, 1e+16, (20,), float64), Box(-1e+16, 1e+16, (20,), float64), Box(-1e+16, 1e+16, (20,), float64))

In [4]:
env.action_space

Tuple(Discrete(10), Discrete(10), Discrete(10))

Consequently, actions are passed as a tuple. The step will stop the first time any
satellite completes an action.

In [5]:
observation, reward, terminated, truncated, info = env.step([7, 9, 8])

2026-11-04 04:33:42,990 gym                            INFO       <0.00> === STARTING STEP ===


2026-11-04 04:33:42,991 sats.satellite.EO_1            INFO       <0.00> EO_1: target index 7 tasked


2026-11-04 04:33:42,991 sats.satellite.EO_1            INFO       <0.00> EO_1: Target(tgt-556) tasked for imaging


2026-11-04 04:33:42,992 sats.satellite.EO_1            INFO       <0.00> EO_1: Target(tgt-556) window enabled: 66.4 to 261.6


2026-11-04 04:33:42,992 sats.satellite.EO_1            INFO       <0.00> EO_1: setting timed terminal event at 261.6


2026-11-04 04:33:42,992 sats.satellite.EO_2            INFO       <0.00> EO_2: target index 9 tasked


... (7 outputs left out)

2026-11-04 04:33:43,010 sats.satellite.EO_1            INFO       <77.00> EO_1: imaged Target(tgt-556)


2026-11-04 04:33:43,010 data.base                      INFO       <77.00> Total reward: {'EO_1': 0.8271220322058404}


2026-11-04 04:33:43,011 sats.satellite.EO_1            INFO       <77.00> EO_1: Satellite EO_1 requires retasking


2026-11-04 04:33:43,012 gym                            INFO       <77.00> Step reward: 0.8271220322058404


In [6]:
observation

(array([ 0.71415019, -0.01350877,  0.52884109, -0.01350877,  0.6190034 ,
        -0.01350877,  0.47099679, -0.00812986,  0.95069383,  0.00401149,
         0.13267318,  0.00383247,  0.980529  ,  0.00431542,  0.42272754,
         0.00535964,  0.1972668 ,  0.0439062 ,  0.33214046,  0.04154876]),
 array([ 8.87440349e-01, -1.35087719e-02,  6.79166472e-01, -1.35087719e-02,
         7.17530063e-01, -1.35087719e-02,  5.13973523e-01,  5.65161227e-04,
         4.46327722e-01,  9.60351597e-03,  7.97557091e-01,  1.47596517e-02,
         4.42840784e-01,  1.04417971e-02,  7.74234520e-01,  4.97451454e-02,
         5.41066360e-01,  5.11585208e-02,  4.56829073e-01,  7.58755392e-02]),
 array([ 0.75127807, -0.01350877,  0.02898462, -0.01350877,  0.1535117 ,
         0.00673973,  0.42335734,  0.00862421,  0.69786774,  0.02131506,
         0.73626319,  0.03619369,  0.3405966 ,  0.03192324,  0.12162146,
         0.04324165,  0.3436594 ,  0.05869828,  0.22235775,  0.05885203]))

At this point, either every satellite can be retasked, or satellites can continue their
previous action by passing `None` as the action. To see which satellites must be
retasked (i.e. their previous action is done and they have nothing more to do), look at
`"requires_retasking"` in each satellite's info.

In [7]:
info

{'EO_1': {'requires_retasking': True},
 'EO_2': {'requires_retasking': False},
 'EO_3': {'requires_retasking': False},
 'd_ts': 77.0}

Based on this list, we decide here to only retask the satellite that needs it.

In [8]:
actions = [
    0 if info[sat.name]["requires_retasking"] else None
    for sat in env.unwrapped.satellites
]
actions

[0, None, None]

In [9]:
observation, reward, terminated, truncated, info = env.step(actions)

2026-11-04 04:33:43,024 gym                            INFO       <77.00> === STARTING STEP ===


2026-11-04 04:33:43,024 sats.satellite.EO_1            INFO       <77.00> EO_1: target index 0 tasked


2026-11-04 04:33:43,024 sats.satellite.EO_1            INFO       <77.00> EO_1: Target(tgt-110) tasked for imaging


2026-11-04 04:33:43,025 sats.satellite.EO_1            INFO       <77.00> EO_1: Target(tgt-110) window enabled: 0.0 to 119.6


2026-11-04 04:33:43,025 sats.satellite.EO_1            INFO       <77.00> EO_1: setting timed terminal event at 119.6


2026-11-04 04:33:43,033 sats.satellite.EO_1            INFO       <120.00> EO_1: timed termination at 119.6 for Target(tgt-110) window


2026-11-04 04:33:43,033 data.base                      INFO       <120.00> Total reward: {}


2026-11-04 04:33:43,034 sats.satellite.EO_1            INFO       <120.00> EO_1: Satellite EO_1 requires retasking


2026-11-04 04:33:43,034 sats.satellite.EO_2            INFO       <120.00> EO_2: Finding opportunity windows from 600.00 to 1200.00 seconds


2026-11-04 04:33:43,050 gym                            INFO       <120.00> Step reward: 0.0


In this environment, the environment will stop if any agent dies. To demonstrate this,
one satellite is forcibly killed.

In [10]:
from Basilisk.architecture import messaging


def isnt_alive(log_failure=False):
    """Mock satellite 0 dying."""
    self = env.unwrapped.satellites[0]
    death_message = messaging.PowerStorageStatusMsgPayload()
    death_message.storageLevel = 0.0
    self.dynamics.powerMonitor.batPowerOutMsg.write(death_message)
    return self.dynamics.is_alive(log_failure=log_failure) and self.fsw.is_alive(
        log_failure=log_failure
    )


env.unwrapped.satellites[0].is_alive = isnt_alive
observation, reward, terminated, truncated, info = env.step([6, 7, 9])

2026-11-04 04:33:43,053 gym                            INFO       <120.00> === STARTING STEP ===


2026-11-04 04:33:43,054 sats.satellite.EO_1            INFO       <120.00> EO_1: target index 6 tasked


2026-11-04 04:33:43,054 sats.satellite.EO_1            INFO       <120.00> EO_1: Target(tgt-128) tasked for imaging


2026-11-04 04:33:43,054 sats.satellite.EO_1            INFO       <120.00> EO_1: Target(tgt-128) window enabled: 107.5 to 315.3


2026-11-04 04:33:43,054 sats.satellite.EO_1            INFO       <120.00> EO_1: setting timed terminal event at 315.3


2026-11-04 04:33:43,055 sats.satellite.EO_2            INFO       <120.00> EO_2: target index 7 tasked


... (10 outputs left out)

2026-11-04 04:33:43,070 sats.satellite.EO_1            WARNING    <183.00> EO_1: failed battery_valid check


2026-11-04 04:33:43,071 gym                            INFO       <183.00> Step reward: -0.5772724593844423


2026-11-04 04:33:43,071 gym                            INFO       <183.00> Episode terminated: True


2026-11-04 04:33:43,071 gym                            INFO       <183.00> Episode truncated: False


## PettingZoo API

The [PettingZoo parallel API](https://pettingzoo.farama.org/api/parallel/) environment, 
ConstellationTasking, is largely the same as GeneralSatelliteTasking. See their
documentation for a full description of the API. It tends to separate things into
dictionaries keyed by agent, rather than tuples.

In [11]:
from bsk_rl import ConstellationTasking

env = ConstellationTasking(
    satellites=[
        ImagingSatellite("EO_1", sat_args),
        ImagingSatellite("EO_2", sat_args),
        ImagingSatellite("EO_3", sat_args),
    ],
    scenario=scene.UniformTargets(1000),
    rewarder=data.UniqueImageReward(),
    communicator=comm.LOSCommunication(),  # Note that dyn must inherit from LOSCommunication
    sat_arg_randomizer=sat_arg_randomizer,
    log_level="INFO",
)
env.reset()

env.observation_spaces

2026-11-04 04:33:43,075                                WARNING    Creating logger for new env on PID=4576. Old environments in process may now log times incorrectly.


2026-11-04 04:33:43,076 gym                            INFO       Resetting environment with seed=4217472450


2026-11-04 04:33:43,077 scene.targets                  INFO       Generating 1000 targets


2026-11-04 04:33:43,108 sats.satellite.EO_1            INFO       <0.00> EO_1: Finding opportunity windows from 0.00 to 600.00 seconds


2026-11-04 04:33:43,122 sats.satellite.EO_2            INFO       <0.00> EO_2: Finding opportunity windows from 0.00 to 600.00 seconds


2026-11-04 04:33:43,137 sats.satellite.EO_2            INFO       <0.00> EO_2: Finding opportunity windows from 600.00 to 1200.00 seconds


2026-11-04 04:33:43,154 sats.satellite.EO_3            INFO       <0.00> EO_3: Finding opportunity windows from 0.00 to 600.00 seconds


2026-11-04 04:33:43,170 gym                            INFO       <0.00> Environment reset


{'EO_1': Box(-1e+16, 1e+16, (20,), float64),
 'EO_2': Box(-1e+16, 1e+16, (20,), float64),
 'EO_3': Box(-1e+16, 1e+16, (20,), float64)}

In [12]:
env.action_spaces

{'EO_1': Discrete(10), 'EO_2': Discrete(10), 'EO_3': Discrete(10)}

Actions are passed as a dictionary; the agent names can be accessed through the `agents`
property.

In [13]:
observation, reward, terminated, truncated, info = env.step(
    {
        env.agents[0]: 7,
        env.agents[1]: 9,
        env.agents[2]: 8,
    }
)

2026-11-04 04:33:43,177 gym                            INFO       <0.00> === STARTING STEP ===


2026-11-04 04:33:43,178 sats.satellite.EO_1            INFO       <0.00> EO_1: target index 7 tasked


2026-11-04 04:33:43,178 sats.satellite.EO_1            INFO       <0.00> EO_1: Target(tgt-60) tasked for imaging


2026-11-04 04:33:43,179 sats.satellite.EO_1            INFO       <0.00> EO_1: Target(tgt-60) window enabled: 292.0 to 495.7


2026-11-04 04:33:43,179 sats.satellite.EO_1            INFO       <0.00> EO_1: setting timed terminal event at 495.7


2026-11-04 04:33:43,180 sats.satellite.EO_2            INFO       <0.00> EO_2: target index 9 tasked


... (9 outputs left out)

2026-11-04 04:33:43,236 sats.satellite.EO_1            INFO       <295.00> EO_1: Satellite EO_1 requires retasking


2026-11-04 04:33:43,237 sats.satellite.EO_1            INFO       <295.00> EO_1: Finding opportunity windows from 600.00 to 1200.00 seconds


2026-11-04 04:33:43,252 sats.satellite.EO_3            INFO       <295.00> EO_3: Finding opportunity windows from 600.00 to 1200.00 seconds


2026-11-04 04:33:43,277 gym                            INFO       <295.00> Step reward: {'EO_1': 0.8852183995254183}


In [14]:
observation

{'EO_1': array([ 0.01044845, -0.03319852,  0.51119378,  0.03625276,  0.88005098,
         0.04511077,  0.38303535,  0.05844256,  0.81097564,  0.05996345,
         0.82297049,  0.06891204,  0.85354997,  0.07675709,  0.53605236,
         0.10182811,  0.98103156,  0.07912379,  0.29069639,  0.11528399]),
 'EO_2': array([0.93521359, 0.00316664, 0.08116887, 0.00299972, 0.30310937,
        0.0265245 , 0.03807943, 0.03786231, 0.54471386, 0.01792049,
        0.03358045, 0.04298473, 0.35670589, 0.06228091, 0.8952031 ,
        0.03873192, 0.41141777, 0.05424434, 0.83072889, 0.05242561]),
 'EO_3': array([ 0.13509072, -0.02764596,  0.05505364, -0.0074827 ,  0.5618506 ,
         0.01110323,  0.70467555,  0.00424143,  0.68671858,  0.04192075,
         0.16838453,  0.01535594,  0.49743994,  0.02817894,  0.17425033,
         0.05508725,  0.4908006 ,  0.05118064,  0.03698582,  0.05186847])}

Other than compatibility with MARL algorithms, the main benefit of the PettingZoo API
is that it allows for individual agents to fail without terminating the entire environment.

In [15]:
# Immediately kill satellite 0
env.unwrapped.satellites[0].is_alive = isnt_alive
env.agents

['EO_1', 'EO_2', 'EO_3']

In [16]:
observation, reward, terminated, truncated, info = env.step(
    {
        env.agents[0]: 7,
        env.agents[1]: 9,
    }
)

2026-11-04 04:33:43,286 gym                            INFO       <295.00> === STARTING STEP ===


2026-11-04 04:33:43,286 sats.satellite.EO_1            INFO       <295.00> EO_1: target index 7 tasked


2026-11-04 04:33:43,287 sats.satellite.EO_1            INFO       <295.00> EO_1: Target(tgt-853) tasked for imaging


2026-11-04 04:33:43,287 sats.satellite.EO_1            INFO       <295.00> EO_1: Target(tgt-853) window enabled: 875.4 to 924.5


2026-11-04 04:33:43,287 sats.satellite.EO_1            INFO       <295.00> EO_1: setting timed terminal event at 924.5


2026-11-04 04:33:43,288 sats.satellite.EO_2            INFO       <295.00> EO_2: target index 9 tasked


... (4 outputs left out)

2026-11-04 04:33:43,295 data.base                      INFO       <322.00> Total reward: {'EO_3': 0.7046755536099801}


2026-11-04 04:33:43,295 sats.satellite.EO_3            INFO       <322.00> EO_3: Satellite EO_3 requires retasking


2026-11-04 04:33:43,297 gym                            INFO       <322.00> Step reward: {'EO_1': -1.0, 'EO_3': 0.7046755536099801}


2026-11-04 04:33:43,297 gym                            INFO       <322.00> Episode terminated: ['EO_1']
